## Step 1 : Load the Existing Chunk-Level Vector Store

The original RAG experiment created 67,303 patent chunks and stored
their embeddings in ChromaDB.

The existing ChromaDB collection is available, so we reuse it directly
instead of recreating the chunks and embeddings.

This allows the improved RAG pipeline to focus on retrieval,
cross-encoder reranking, and later LLM-based generation.

## Step 2 : Load the Existing Chunk-Level Vector Store

The existing ChromaDB collection contains the 67,303 patent chunks
created in the previous RAG experiment.

We reuse these stored embeddings for the improved retrieval pipeline.

In [1]:
import chromadb

client = chromadb.PersistentClient(
    path="../data/chroma_db"
)

chunk_collection = client.get_collection(
    name="patent_rag_chunks"
)

print("Collection loaded.")
print("Number of chunks:", chunk_collection.count())

Collection loaded.
Number of chunks: 67303


## Step 3 : Define the Search Query

We use the same query from the previous RAG experiment so that the
new retrieval and reranking results can be compared with the original
results.

The query describes a device for holding cylindrical components during
surface treatment while minimizing contact with the component.

In [2]:
query = """
A device for holding cylindrical components during surface treatment
while minimizing the contact area with the component.
"""


print(query)


A device for holding cylindrical components during surface treatment
while minimizing the contact area with the component.



## Step 4 : Load the Embedding Model

The existing ChromaDB collection was created using the
`all-MiniLM-L6-v2` embedding model.

We load the same model so that the query embedding is compatible
with the stored 384-dimensional embeddings.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384


/tmp/ipykernel_6625/272882491.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [4]:

query_embedding = model.encode(query).tolist()

results = chunk_collection.query(
    query_embeddings=[query_embedding],
    n_results=50,
    include=["documents", "metadatas", "distances"]
)

print("Retrieved chunks:", len(results["documents"][0]))

Retrieved chunks: 50


## Step 5 : Inspect the Initial Retrieval

The embedding model retrieves the 50 most similar chunks from the
patent corpus.

We inspect the top 10 candidates before reranking so that the
cross-encoder results can later be compared with the original ranking.

In [5]:
print(results["metadatas"][0][0])

{'decision': 'PENDING', 'patent_number': '14904285', 'chunk_index': 0, 'title': 'SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PARTS, LIKE LANDING GEAR RODS AND CYLINDERS'}


In [6]:
for i in range(10):
    print(f"\nRank {i + 1}")
    print("Patent:", results["metadatas"][0][i]["patent_number"])
    print("Chunk:", results["metadatas"][0][i]["chunk_index"])
    print("Distance:", results["distances"][0][i])
    print("Text:", results["documents"][0][i][:500])


Rank 1
Patent: 14904285
Chunk: 0
Distance: 0.7893787622451782
Text: supporting tool for supporting cylindrical parts, like landing gear rods and cylinders supporting tool ( 1 ) for supporting a substantially cylindrical part, like a landing gear rod or cylinder, said supporting tool comprising a metallic frame carrying at least one metallic ring ( 3 ) in electrical conductive relation with the frame, wherein the metallic ring carries a plurality of conical spikes ( 6 ) of non - conductive material projecting inward so as to engage the part with limited contact s

Rank 2
Patent: 14989141
Chunk: 6
Distance: 0.8476866483688354
Text: exposed surface area exhibiting a lower static coefficient of friction than that of the higher friction area, wherein the lower friction layer is operatively adapted so as to no longer cover a substantial portion of the higher friction area, after the structure has been used to mount a pollution control element in a housing of a pollution control device. it c

## Step 6 : Convert Distances to Similarities

ChromaDB returns distances for the retrieved chunks.

The patent-level scoring method from the previous RAG experiment
uses similarity scores, so we convert the distances to similarities
before applying the same patent-level ranking formula.

In [8]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

retrieved_embeddings = model.encode(
    results["documents"][0],
    convert_to_numpy=True
)

query_embedding_array = np.array(query_embedding).reshape(1, -1)

similarities = cosine_similarity(
    query_embedding_array,
    retrieved_embeddings
)[0]

print("Top similarity:", similarities[0])

Top similarity: 0.6053105741917703


## Step 7 : Group Retrieved Chunks by Patent

The initial retrieval returns chunks rather than complete patents.

Multiple retrieved chunks may belong to the same patent, so we group
the 50 retrieved chunks by patent before calculating the patent-level
score.

In [9]:
from collections import defaultdict

patent_chunks = defaultdict(list)

for i in range(50):
    patent_number = results["metadatas"][0][i]["patent_number"]

    patent_chunks[patent_number].append({
        "chunk_index": results["metadatas"][0][i]["chunk_index"],
        "similarity": similarities[i],
        "text": results["documents"][0][i]
    })

print("Retrieved chunks:", 50)
print("Unique patents:", len(patent_chunks))

Retrieved chunks: 50
Unique patents: 27


## Step 8 : Calculate Patent-Level Scores

For each retrieved patent, we calculate a patent-level relevance
score using the same method as the previous RAG experiment:

Patent Score = 0.5 × Max Similarity
             + 0.5 × Top-3 Mean Similarity

If fewer than three chunks are retrieved for a patent, all available
chunks are used for the mean.

In [10]:
patent_scores = {}

for patent_number, chunks in patent_chunks.items():
    chunk_similarities = sorted(
        [chunk["similarity"] for chunk in chunks],
        reverse=True
    )

    max_similarity = chunk_similarities[0]
    top_3_mean = sum(chunk_similarities[:3]) / min(3, len(chunk_similarities))

    patent_score = 0.5 * max_similarity + 0.5 * top_3_mean

    patent_scores[patent_number] = {
        "max_similarity": max_similarity,
        "top_3_mean": top_3_mean,
        "patent_score": patent_score,
        "num_chunks": len(chunks)
    }

print("Patent scores calculated:", len(patent_scores))

Patent scores calculated: 27


## Step 9 : Rank Patents

The patents are ranked using the patent-level score calculated in
Step 8.

This produces the baseline patent ranking that will later be compared
with the reranked results from the cross-encoder.

In [11]:
ranked_patents = sorted(
    patent_scores.items(),
    key=lambda x: x[1]["patent_score"],
    reverse=True
)

print("Top 5 patents:\n")

for rank, (patent_number, scores) in enumerate(ranked_patents[:5], start=1):
    print(
        f"{rank}. Patent {patent_number} | "
        f"Score: {scores['patent_score']:.4f} | "
        f"Max: {scores['max_similarity']:.4f} | "
        f"Top-3 Mean: {scores['top_3_mean']:.4f} | "
        f"Chunks: {scores['num_chunks']}"
    )

Top 5 patents:

1. Patent 14904285 | Score: 0.5977 | Max: 0.6053 | Top-3 Mean: 0.5901 | Chunks: 2
2. Patent 14989141 | Score: 0.5618 | Max: 0.5762 | Top-3 Mean: 0.5474 | Chunks: 4
3. Patent 14988948 | Score: 0.5518 | Max: 0.5650 | Top-3 Mean: 0.5387 | Chunks: 4
4. Patent 14904839 | Score: 0.5495 | Max: 0.5638 | Top-3 Mean: 0.5351 | Chunks: 4
5. Patent 14992023 | Score: 0.5346 | Max: 0.5528 | Top-3 Mean: 0.5164 | Chunks: 3


## Step 10 : Load the Cross-Encoder Reranker

The embedding model performs the initial fast retrieval of the top 50
candidate chunks.

A cross-encoder is then used to evaluate the relevance of each
query–chunk pair more precisely.

Unlike the embedding model, the cross-encoder receives the query and
candidate chunk together and produces a relevance score.

In [12]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded.


## Step 11 : Rerank the Retrieved Chunks

The cross-encoder evaluates each query–chunk pair and assigns a
relevance score.

We use the 50 chunks retrieved by the embedding model as candidates.

In [13]:
pairs = [
    (query, document)
    for document in results["documents"][0]
]

reranker_scores = reranker.predict(pairs)

print("Reranked chunks:", len(reranker_scores))
print("Top reranker score:", reranker_scores.max())

Reranked chunks: 50
Top reranker score: 2.0979903


## Step 12 : Inspect the Reranked Chunks

The cross-encoder has assigned a relevance score to each of the
50 retrieved chunks.

We sort the chunks by this score to obtain the reranked order.

In [14]:
reranked_indices = sorted(
    range(len(reranker_scores)),
    key=lambda i: reranker_scores[i],
    reverse=True
)

print("Top 10 reranked chunks:\n")

for rank, i in enumerate(reranked_indices[:10], start=1):
    metadata = results["metadatas"][0][i]

    print(
        f"Rank {rank} | "
        f"Patent: {metadata['patent_number']} | "
        f"Chunk: {metadata['chunk_index']} | "
        f"Score: {reranker_scores[i]:.4f}"
    )

Top 10 reranked chunks:

Rank 1 | Patent: 14904285 | Chunk: 0 | Score: 2.0980
Rank 2 | Patent: 14993609 | Chunk: 4 | Score: -3.3710
Rank 3 | Patent: 15001440 | Chunk: 0 | Score: -3.4965
Rank 4 | Patent: 14902717 | Chunk: 8 | Score: -3.5018
Rank 5 | Patent: 14905370 | Chunk: 7 | Score: -3.6564
Rank 6 | Patent: 14904285 | Chunk: 1 | Score: -4.2190
Rank 7 | Patent: 14988948 | Chunk: 18 | Score: -4.3949
Rank 8 | Patent: 14902717 | Chunk: 22 | Score: -4.5366
Rank 9 | Patent: 14905370 | Chunk: 18 | Score: -4.7440
Rank 10 | Patent: 14904113 | Chunk: 6 | Score: -5.0850


### Step 12: Inspect the Reranked Chunks

The cross-encoder assigns a relevance score to each query–chunk pair and we sort the chunks by this score.

The cross-encoder scores are **raw relevance scores (logits)**, not normalized similarity values. Therefore, they are not restricted to the range 0–1 and may be negative.

A higher score indicates that the cross-encoder considers the query and chunk more relevant to each other, while a lower or negative score indicates weaker relevance.

These scores should therefore be used for **relative ranking** rather than interpreted as probabilities or percentage similarities.

For example, a score of `2.10` is considered more relevant than `-3.37`, but `-3.37` does not mean that the chunk has “negative similarity.”

In [15]:
from scipy.special import expit

probability_like_scores = expit(reranker_scores)

### Step 13: Calculate Reranked Patent-Level Scores

The cross-encoder has reranked the 50 candidate chunks using its relevance scores.

We now group these reranked chunks by patent and apply the same patent-level scoring strategy used in the baseline:

Patent Score = 0.5 × Max Reranker Score + 0.5 × Top-3 Mean Reranker Score

This allows us to compare the patent ranking produced by the original embedding-based retrieval with the ranking produced after cross-encoder reranking.

In [16]:
from collections import defaultdict

reranked_patent_chunks = defaultdict(list)

for i in reranked_indices:
    patent_number = results["metadatas"][0][i]["patent_number"]

    reranked_patent_chunks[patent_number].append({
        "chunk_index": results["metadatas"][0][i]["chunk_index"],
        "reranker_score": reranker_scores[i],
        "text": results["documents"][0][i]
    })

reranked_patent_scores = {}

for patent_number, chunks in reranked_patent_chunks.items():
    chunk_scores = sorted(
        [chunk["reranker_score"] for chunk in chunks],
        reverse=True
    )

    max_score = chunk_scores[0]
    top_3_mean = sum(chunk_scores[:3]) / min(3, len(chunk_scores))

    patent_score = 0.5 * max_score + 0.5 * top_3_mean

    reranked_patent_scores[patent_number] = {
        "max_score": max_score,
        "top_3_mean": top_3_mean,
        "patent_score": patent_score,
        "num_chunks": len(chunks)
    }

print("Reranked patent scores calculated:", len(reranked_patent_scores))

Reranked patent scores calculated: 27


### Step 14: Rank Patents After Reranking

The patents are now ranked using the patent-level scores calculated from the cross-encoder relevance scores.

This produces the final patent ranking after reranking, which can be compared directly with the baseline ranking from Step 9.

In [17]:
ranked_reranked_patents = sorted(
    reranked_patent_scores.items(),
    key=lambda x: x[1]["patent_score"],
    reverse=True
)

print("Top 5 patents after reranking:\n")

for rank, (patent_number, scores) in enumerate(
    ranked_reranked_patents[:5], start=1
):
    print(
        f"{rank}. Patent {patent_number} | "
        f"Score: {scores['patent_score']:.4f} | "
        f"Max: {scores['max_score']:.4f} | "
        f"Top-3 Mean: {scores['top_3_mean']:.4f} | "
        f"Chunks: {scores['num_chunks']}"
    )

Top 5 patents after reranking:

1. Patent 14904285 | Score: 0.5188 | Max: 2.0980 | Top-3 Mean: -1.0605 | Chunks: 2
2. Patent 14993609 | Score: -3.3710 | Max: -3.3710 | Top-3 Mean: -3.3710 | Chunks: 1
3. Patent 15001440 | Score: -3.4965 | Max: -3.4965 | Top-3 Mean: -3.4965 | Chunks: 1
4. Patent 14902717 | Score: -3.7605 | Max: -3.5018 | Top-3 Mean: -4.0192 | Chunks: 2
5. Patent 14905370 | Score: -3.9283 | Max: -3.6564 | Top-3 Mean: -4.2002 | Chunks: 2


### Step 15: Inspect the Top Reranked Chunks

We inspect the highest-ranked chunks after cross-encoder reranking.

The purpose is to determine whether the reranker improves the semantic relevance of the retrieved evidence compared with the original embedding-based ranking.

The reranker score is used only for ordering the chunks at this stage. We inspect the actual patent text to evaluate whether the new ranking is more relevant to the query.

In [18]:
print("Top 10 reranked chunks:\n")

for rank, i in enumerate(reranked_indices[:10], start=1):
    metadata = results["metadatas"][0][i]

    print(f"\nRank {rank}")
    print("Patent:", metadata["patent_number"])
    print("Chunk:", metadata["chunk_index"])
    print("Reranker score:", f"{reranker_scores[i]:.4f}")
    print("Text:", results["documents"][0][i][:700])

Top 10 reranked chunks:


Rank 1
Patent: 14904285
Chunk: 0
Reranker score: 2.0980
Text: supporting tool for supporting cylindrical parts, like landing gear rods and cylinders supporting tool ( 1 ) for supporting a substantially cylindrical part, like a landing gear rod or cylinder, said supporting tool comprising a metallic frame carrying at least one metallic ring ( 3 ) in electrical conductive relation with the frame, wherein the metallic ring carries a plurality of conical spikes ( 6 ) of non - conductive material projecting inward so as to engage the part with limited contact surface when said part is in place within the tool, said ring being openable so as to allow the part to be inserted into the ring and closable after the part has been inserted. technological background

Rank 2
Patent: 14993609
Chunk: 4
Reranker score: -3.3710
Text: port, a manifold and other means of controlling the flow of fluid inside the extravasation minimization device. the distal tip of the atraumatic ex

### Step 16: Compare Embedding and Reranker Rankings

We compare the original embedding-based ranking with the cross-encoder reranking for the same 50 candidate chunks.

This allows us to see how the reranker changes the ordering of the retrieved evidence.

We compare:
- Original embedding rank and cosine similarity
- Cross-encoder rank and reranker score
- Patent and chunk identity

The scores are not directly comparable because they come from different models and scales. We are comparing their **rankings**, not their numerical values.

In [21]:
import pandas as pd
comparison = []

for embedding_rank, i in enumerate(range(len(results["documents"][0])), start=1):
    metadata = results["metadatas"][0][i]

    comparison.append({
        "patent": metadata["patent_number"],
        "chunk": metadata["chunk_index"],
        "embedding_rank": embedding_rank,
        "embedding_similarity": similarities[i],
        "reranker_rank": list(reranked_indices).index(i) + 1,
        "reranker_score": reranker_scores[i]
    })

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    "reranker_rank"
).reset_index(drop=True)

comparison_df.head(10)

,patent,chunk,embedding_rank,embedding_similarity,reranker_rank,reranker_score
0,14904285,0,1,0.605311,1,2.097990
1,14993609,4,45,0.485870,2,-3.371048
2,15001440,0,21,0.505265,3,-3.496501
3,14902717,8,22,0.504185,4,-3.501766
4,14905370,7,16,0.518354,5,-3.656370
5,14904285,1,3,0.574815,6,-4.218959
6,14988948,18,6,0.556245,7,-4.394887
7,14902717,22,39,0.490102,8,-4.536611
8,14905370,18,46,0.485095,9,-4.743974
9,14904113,6,8,0.540174,10,-5.085048


### Step 17: Manually Evaluate Reranked Chunks

We now manually evaluate the top reranked chunks for relevance to the query.

A chunk is labeled:

- Relevant — directly addresses the requested cylindrical holding/supporting device.
- Partially Relevant — related to the general concept but does not directly address the requested device.
- Irrelevant — does not meaningfully address the query.

These labels allow us to compare the actual relevance of the original embedding ranking with the cross-encoder ranking.

In [22]:
# Display the top 10 reranked chunks for manual evaluation

for rank, i in enumerate(reranked_indices[:10], start=1):
    metadata = results["metadatas"][0][i]

    print(f"\n{'='*80}")
    print(f"Rank: {rank}")
    print(f"Patent: {metadata['patent_number']}")
    print(f"Chunk: {metadata['chunk_index']}")
    print(f"Reranker score: {reranker_scores[i]:.4f}")
    print(f"\nText:\n{results['documents'][0][i][:1000]}")


Rank: 1
Patent: 14904285
Chunk: 0
Reranker score: 2.0980

Text:
supporting tool for supporting cylindrical parts, like landing gear rods and cylinders supporting tool ( 1 ) for supporting a substantially cylindrical part, like a landing gear rod or cylinder, said supporting tool comprising a metallic frame carrying at least one metallic ring ( 3 ) in electrical conductive relation with the frame, wherein the metallic ring carries a plurality of conical spikes ( 6 ) of non - conductive material projecting inward so as to engage the part with limited contact surface when said part is in place within the tool, said ring being openable so as to allow the part to be inserted into the ring and closable after the part has been inserted. technological background supporting tools are known for supporting parts during specific treatments, like electrolytic coating. such tools must be designed to support the part with minimal contacting surface to allow the near totality of the surface of the pa

### Step 18: Evaluate Retrieval Precision

We evaluate the top-k results using manually assigned relevance labels.

For this query, patent 14904285 is the relevant patent. A retrieved chunk is considered relevant if it belongs to this patent.

We calculate Precision@K for the original embedding ranking and the cross-encoder reranked ranking.

Precision@K measures the proportion of relevant results among the top K retrieved chunks.

In [23]:
relevant_patent = "14904285"

def precision_at_k(ranking_indices, k):
    top_k = ranking_indices[:k]

    relevant = sum(
        results["metadatas"][0][i]["patent_number"] == relevant_patent
        for i in top_k
    )

    return relevant / k


for k in [5, 10, 20, 50]:
    embedding_precision = precision_at_k(
        list(range(50)), k
    )

    reranker_precision = precision_at_k(
        reranked_indices, k
    )

    print(
        f"Precision@{k}: "
        f"Embedding = {embedding_precision:.2f}, "
        f"Reranker = {reranker_precision:.2f}"
    )

Precision@5: Embedding = 0.40, Reranker = 0.20
Precision@10: Embedding = 0.20, Reranker = 0.20
Precision@20: Embedding = 0.10, Reranker = 0.10
Precision@50: Embedding = 0.04, Reranker = 0.04


### Step 19: Evaluate Retrieval Recall

We now calculate Recall@K for the original embedding retrieval and the cross-encoder reranking.

Recall@K measures whether the relevant patent appears within the top K retrieved chunks.

For this experiment, patent 14904285 is treated as the relevant patent.

In [24]:
def recall_at_k(ranking_indices, k):
    top_k = ranking_indices[:k]

    return int(
        any(
            results["metadatas"][0][i]["patent_number"] == relevant_patent
            for i in top_k
        )
    )


for k in [5, 10, 20, 50]:
    embedding_recall = recall_at_k(
        list(range(50)), k
    )

    reranker_recall = recall_at_k(
        reranked_indices, k
    )

    print(
        f"Recall@{k}: "
        f"Embedding = {embedding_recall:.2f}, "
        f"Reranker = {reranker_recall:.2f}"
    )

Recall@5: Embedding = 1.00, Reranker = 1.00
Recall@10: Embedding = 1.00, Reranker = 1.00
Recall@20: Embedding = 1.00, Reranker = 1.00
Recall@50: Embedding = 1.00, Reranker = 1.00


### Step 20: Evaluate Patent-Level Recall

The final retrieval system ranks patents rather than individual chunks.

We therefore evaluate whether the relevant patent appears within the top-k unique patents for both the original embedding-based retrieval and the cross-encoder reranked retrieval.

This evaluation is aligned with the patent-level retrieval architecture.


In [25]:
def unique_patents_from_ranking(ranking_indices, k):
    patents = []

    for i in ranking_indices:
        patent = results["metadatas"][0][i]["patent_number"]

        if patent not in patents:
            patents.append(patent)

        if len(patents) == k:
            break

    return patents


for k in [5, 10, 20]:
    embedding_patents = unique_patents_from_ranking(
        list(range(50)), k
    )

    reranker_patents = unique_patents_from_ranking(
        reranked_indices, k
    )

    embedding_recall = int(
        relevant_patent in embedding_patents
    )

    reranker_recall = int(
        relevant_patent in reranker_patents
    )

    print(
        f"Patent Recall@{k}: "
        f"Embedding = {embedding_recall:.2f}, "
        f"Reranker = {reranker_recall:.2f}"
    )

Patent Recall@5: Embedding = 1.00, Reranker = 1.00
Patent Recall@10: Embedding = 1.00, Reranker = 1.00
Patent Recall@20: Embedding = 1.00, Reranker = 1.00


### Step 21: Compare Patent-Level Rankings

We compare the unique patent rankings produced by the original embedding retrieval and the cross-encoder reranking.

This shows whether reranking changes the position of the relevant patent at the patent level.

In [26]:
embedding_patents = unique_patents_from_ranking(
    list(range(50)), 20
)

reranker_patents = unique_patents_from_ranking(
    reranked_indices, 20
)

print("Embedding ranking:")
for rank, patent in enumerate(embedding_patents, start=1):
    print(f"{rank}. {patent}")

print("\nReranker ranking:")
for rank, patent in enumerate(reranker_patents, start=1):
    print(f"{rank}. {patent}")

Embedding ranking:
1. 14904285
2. 14989141
3. 14988948
4. 14904839
5. 14992023
6. 14904113
7. 14904987
8. 14905300
9. 14996751
10. 14905370
11. 14904001
12. 14993597
13. 14990813
14. 15001440
15. 14902717
16. 14991697
17. 14988231
18. 14988443
19. 14988720
20. 14993561

Reranker ranking:
1. 14904285
2. 14993609
3. 15001440
4. 14902717
5. 14905370
6. 14988948
7. 14904113
8. 14904839
9. 14993852
10. 14904987
11. 14996751
12. 15001644
13. 14994259
14. 14990813
15. 14989141
16. 14988443
17. 14904771
18. 14993597
19. 14991697
20. 14904001


In [27]:
import subprocess

result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True
)

print(result.stdout)

NAME                       ID              SIZE      MODIFIED    
llama3.2:3b                a80c4f17acd5    2.0 GB    4 days ago     
gemma3:4b                  a2af6cc3eb7f    3.3 GB    4 weeks ago    
deepseek-r1:8b             6995872bfe4c    5.2 GB    4 weeks ago    
qwen3.5:4b                 2a654d98e6fb    3.4 GB    4 weeks ago    
nomic-embed-text:latest    0a109f422b47    274 MB    6 weeks ago    
qwen2.5-coder:3b           f72c60cabf62    1.9 GB    6 weeks ago    



In [28]:
top_n = 10

context_chunks = []

for rank, i in enumerate(reranked_indices[:top_n], start=1):

    metadata = results["metadatas"][0][i]
    document = results["documents"][0][i]

    context_chunks.append(
        f"""
Source {rank}
Patent Number: {metadata["patent_number"]}
Chunk: {metadata["chunk_index"]}

{document}
"""
    )

context = "\n".join(context_chunks)

print(context[:5000])


Source 1
Patent Number: 14904285
Chunk: 0

supporting tool for supporting cylindrical parts, like landing gear rods and cylinders supporting tool ( 1 ) for supporting a substantially cylindrical part, like a landing gear rod or cylinder, said supporting tool comprising a metallic frame carrying at least one metallic ring ( 3 ) in electrical conductive relation with the frame, wherein the metallic ring carries a plurality of conical spikes ( 6 ) of non - conductive material projecting inward so as to engage the part with limited contact surface when said part is in place within the tool, said ring being openable so as to allow the part to be inserted into the ring and closable after the part has been inserted. technological background supporting tools are known for supporting parts during specific treatments, like electrolytic coating. such tools must be designed to support the part with minimal contacting surface to allow the near totality of the surface of the part to be accessible f

In [32]:
prompt = f"""
You are a patent research assistant.

Answer the user's query using ONLY the patent evidence provided below.

User query:
{query}

Retrieved patent evidence:
{context}

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent patent claims, technical details, or conclusions.
3. Do not say that a patent contains a specific claim unless that claim text
   is explicitly present in the provided evidence.
4. If the retrieved evidence only contains a description or abstract, describe
   it as such.
5. Identify patents using their patent numbers.
6. Clearly distinguish between what the evidence states and what is an inference.
7. If the evidence is insufficient, say:
   "The retrieved evidence is insufficient to establish this."
8. Keep the answer concise.

Format the answer as:

Answer:
<direct answer based only on the evidence>

Most relevant patents:
- Patent <number>: <why it is relevant>
- Patent <number>: <why it is relevant>

Evidence:
- Patent <number>, Chunk <number>: <brief evidence>
"""

In [33]:
print(prompt[:8000])


You are a patent research assistant.

Answer the user's query using ONLY the patent evidence provided below.

User query:

A device for holding cylindrical components during surface treatment
while minimizing the contact area with the component.


Retrieved patent evidence:

Source 1
Patent Number: 14904285
Chunk: 0

supporting tool for supporting cylindrical parts, like landing gear rods and cylinders supporting tool ( 1 ) for supporting a substantially cylindrical part, like a landing gear rod or cylinder, said supporting tool comprising a metallic frame carrying at least one metallic ring ( 3 ) in electrical conductive relation with the frame, wherein the metallic ring carries a plurality of conical spikes ( 6 ) of non - conductive material projecting inward so as to engage the part with limited contact surface when said part is in place within the tool, said ring being openable so as to allow the part to be inserted into the ring and closable after the part has been inserted. tech

In [34]:
import subprocess

result = subprocess.run(
    ["ollama", "run", "llama3.2:3b", prompt],
    capture_output=True,
    text=True
)

answer = result.stdout

print(answer)

Answer:
The evidence suggests that a device for holding cylindrical components duri
during surface treatment can be achieved using a supporting tool that minim
minimizes contact area with the component.

Most relevant patents:
- Patent 14904285: This patent describes a supporting tool for cylindrical 
parts, including a metallic frame carrying a metallic ring with conical spi
spikes that engage the part with limited contact surface.
- Patent 14905370: This patent describes a sealing device that includes a c
contact member with a protruding portion that cooperates with a diaphragm s
spring to provide a dual sealing structure.

Evidence:
- Patent 14904285, Chunk 0: This patent describes a supporting tool for cyl
cylindrical parts, including a metallic frame carrying a metallic ring with
with conical spikes that engage the part with limited contact surface.
- Patent 14905370, Chunk 7: This patent describes a sealing device that inc
includes a contact member with a protruding portion that 